# GymScape — Dumbbell detector (train on Open Images, export to TF.js)

Trains a tiny YOLOv8-nano dumbbell detector on Google's **Open Images** `Dumbbell` class
and exports it to **TensorFlow.js** so it runs 100% on-device in the app (no video ever
leaves the phone). ~30–60 min on a free T4 GPU.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.
At the end it downloads `dumbbell_web_model.zip` — send that back and it gets bundled next to MoveNet.


In [ ]:
# 1. Deps (ultralytics for YOLOv8 + tf.js export; fiftyone to pull Open Images)
!pip -q install ultralytics fiftyone
import torch; print('CUDA available:', torch.cuda.is_available())


In [ ]:
# 2. Download the Open Images 'Dumbbell' class (detections) and export to YOLO format
import fiftyone as fo, fiftyone.zoo as foz, os, shutil
CLASSES = ["Dumbbell"]

train = foz.load_zoo_dataset("open-images-v7", split="train",
        label_types=["detections"], classes=CLASSES, only_matching=True, max_samples=4000)
val   = foz.load_zoo_dataset("open-images-v7", split="validation",
        label_types=["detections"], classes=CLASSES, only_matching=True, max_samples=800)
print("train:", len(train), " val:", len(val))

OUT = "/content/db_yolo"
if os.path.exists(OUT): shutil.rmtree(OUT)
for ds, split in [(train, "train"), (val, "val")]:
    ds.export(export_dir=OUT, dataset_type=fo.types.YOLOv5Dataset,
              label_field="ground_truth", classes=CLASSES, split=split)
print("dataset.yaml ->", OUT + "/dataset.yaml")
!cat /content/db_yolo/dataset.yaml


In [ ]:
# 3. Train YOLOv8-nano (single class: dumbbell)
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="/content/db_yolo/dataset.yaml", epochs=50, imgsz=640,
            batch=16, patience=15, name="dumbbell")


In [ ]:
# 4. Quick sanity check on the validation set (mAP)
metrics = YOLO("runs/detect/dumbbell/weights/best.pt").val(data="/content/db_yolo/dataset.yaml")
print("mAP50:", metrics.box.map50, " mAP50-95:", metrics.box.map)


In [ ]:
# 5. Export the trained model to TensorFlow.js (on-device format)
best = YOLO("runs/detect/dumbbell/weights/best.pt")
best.export(format="tfjs", imgsz=640)
WEB = "runs/detect/dumbbell/weights/best_web_model"
print("tf.js model at:", WEB)
!ls -la {WEB}


In [ ]:
# 6. Zip it and download — send this file back to bundle in the app
import shutil
shutil.make_archive("/content/dumbbell_web_model", "zip", "runs/detect/dumbbell/weights/best_web_model")
from google.colab import files
files.download("/content/dumbbell_web_model.zip")
print("Done. If the download does not start, find dumbbell_web_model.zip in the Files panel (left).")
